In [ ]:
!ls /kaggle/input/notebooks/packagemanager/pm-108440548-at-02-01-2026-09-40-45
!pip install --no-index --no-deps \
  /kaggle/input/notebooks/packagemanager/pm-108440548-at-02-01-2026-09-40-45/segmentation_models_pytorch-0.5.0-py3-none-any.whl

In [ ]:
import segmentation_models_pytorch as smp
print(smp.__version__)

In [ ]:
try:
    import cc3d  # connected components in 3D (used in Stage 1 grid detection)
except:         
    #https://pypi.org/project/connected-components-3d/
    #!pip install connected-components-3d

    !ls /kaggle/input/hengck23-demo-submit-physionet/setup
    !pip install connected-components-3d --no-index --find-links=file:///kaggle/input/hengck23-demo-submit-physionet/setup/

import cc3d
import cv2          # OpenCV: all image processing operations
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib
#matplotlib.use('TkAgg')
import shutil

import sys
sys.path.insert(0, '/kaggle/input/datasets/zahouaniyacine/my-stage2-lead-model') #my stage2 attention model added 
sys.path.append('/kaggle/input/hengck23-demo-submit-physionet')
sys.path.append('/kaggle/input/datasets/takashisomeya/physionet-final-submission-models')
# This adds hengck23's pre-packaged code to Python's module search path
# Inside this package:
# - stage0model.py     : the Stage 0 neural network definition
# - stage0common.py    : Stage 0 helper functions (image→batch, output→prediction)
# - stage1model.py     : the Stage 1 neural network definition
# - stage1common.py    : Stage 1 helper functions (output→grid points, rectification)
# - stage2model.py     : Stage 2 model + pixel_to_series conversion
# - stage2common.py    : Stage 2 helper functions

print('import ok!!!')

In [ ]:
import stage2_lead_model
print(stage2_lead_model.__file__)
# Should print: /kaggle/input/my-physionet-stage2/stage2_lead_model.py
# NOT: /kaggle/input/physionet-final-submission-models/stage2_lead_model.py

In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.signal
import scipy.optimize

LEADS = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
MAX_TIME_SHIFT = 0.2
PERFECT_SCORE = 384.0

class ParticipantVisibleError(Exception):
    pass

def compute_power(label: np.ndarray, prediction: np.ndarray):
    if label.ndim != 1 or prediction.ndim != 1:
        raise ParticipantVisibleError("Inputs must be 1-dimensional arrays.")
    finite_mask = np.isfinite(prediction)
    if not np.any(finite_mask):
        raise ParticipantVisibleError("Prediction contains no finite values.")
    prediction = prediction.copy()
    prediction[~finite_mask] = 0
    noise = label - prediction
    p_signal = np.sum(label ** 2)
    p_noise = np.sum(noise ** 2)
    return p_signal, p_noise

def compute_snr(signal_power: float, noise_power: float):
    if noise_power == 0:
        return PERFECT_SCORE
    elif signal_power == 0:
        return 0.0
    else:
        return min(signal_power / noise_power, PERFECT_SCORE)

def align_signals(label: np.ndarray, pred: np.ndarray, max_shift=float("inf")):
    if np.any(~np.isfinite(label)):
        raise ParticipantVisibleError("Label contains non-finite values.")
    if np.sum(np.isfinite(pred)) == 0:
        raise ParticipantVisibleError("Prediction cannot be all NaN/Inf.")

    label_arr = np.asarray(label, dtype=np.float64)
    pred_arr  = np.asarray(pred,  dtype=np.float64)

    label_centered = label_arr - np.mean(label_arr)
    pred_centered  = pred_arr  - np.mean(pred_arr)

    correlation = scipy.signal.correlate(label_centered, pred_centered, mode="full")
    n_label = label_arr.size
    n_pred  = pred_arr.size
    lags = scipy.signal.correlation_lags(n_label, n_pred, mode="full")

    valid_lags_mask = (lags >= -max_shift) & (lags <= max_shift)
    valid_corr = correlation[valid_lags_mask]
    valid_lags = lags[valid_lags_mask]

    max_corr = np.nanmax(valid_corr)
    best_candidates = np.flatnonzero(valid_corr == max_corr)
    best_idx_local = min(best_candidates, key=lambda i: abs(valid_lags[i]))
    time_shift = int(valid_lags[best_idx_local])

    start_padding_len = max(time_shift, 0)
    pred_slice_start  = max(-time_shift, 0)
    pred_slice_end    = min(n_label - time_shift, n_pred)
    end_padding_len   = max(n_label - n_pred - time_shift, 0)

    aligned_pred = np.concatenate([
        np.full(start_padding_len, np.nan),
        pred_arr[pred_slice_start:pred_slice_end],
        np.full(end_padding_len, np.nan)
    ])

    def objective_func(vshift):
        return np.nansum((label_arr - aligned_pred - vshift) ** 2)

    if np.any(np.isfinite(label_arr) & np.isfinite(aligned_pred)):
        result = scipy.optimize.minimize_scalar(objective_func, method="Brent")
        vertical_shift = result.x
        aligned_pred = aligned_pred + vertical_shift

    return aligned_pred

In [ ]:
import importlib 
import torch.nn as nn 
import stage2_lead_model as _m

importlib.reload(_m)

class CrossLeadAttentionFusion(nn.Module):
    def __init__(self, channels, num_leads=4, num_heads=4, dropout=0.1):
        super().__init__()
        self.channels = channels
        self.num_leads = num_leads
        self.num_heads = next((h for h in [8, 4, 2, 1] if channels % h == 0), 1)

        self.norm1 = nn.LayerNorm(channels)
        self.norm2 = nn.LayerNorm(channels)

        self.attn = nn.MultiheadAttention(
            embed_dim=channels,
            num_heads=self.num_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.ffn = nn.Sequential(
            nn.Linear(channels, channels * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(channels * 2, channels),
        )

        self.attn_scale = nn.Parameter(torch.tensor(0.1))
        self.ffn_scale  = nn.Parameter(torch.tensor(0.1))

    def forward(self, x, batch_size=None, batchsize=None):
        if batch_size is None:
            batch_size = batchsize
        if batch_size is None:
            raise ValueError("batch_size must be provided")

        B = batch_size
        _, C, H, W = x.shape

        x_leads = x.view(B, self.num_leads, C, H, W)
        x_hw = x_leads.permute(0, 3, 4, 1, 2)
        x_seq = x_hw.reshape(B * H * W, self.num_leads, C)

        attn_in = self.norm1(x_seq)
        attn_out, _ = self.attn(attn_in, attn_in, attn_in)
        x_seq = x_seq + self.attn_scale * attn_out

        ffn_in = self.norm2(x_seq)
        x_seq = x_seq + self.ffn_scale * self.ffn(ffn_in)

        x_hw = x_seq.reshape(B, H, W, self.num_leads, C)
        x_out = x_hw.permute(0, 3, 4, 1, 2)
        return x_out.reshape(B * self.num_leads, C, H, W)

In [ ]:
MODE   = 'submit'  # submit  local fake
DEVICE = 'cuda'
FLOAT_TYPE = torch.float16 #torch.bfloat16
FAIL_ID = []

KAGGLE_DIR = \
	'/kaggle/input/physionet-ecg-image-digitization'
WEIGHT_DIR = \
	'/kaggle/input/hengck23-demo-submit-physionet/weight'
OUT_DIR = \
    f'/kaggle/working/output-{MODE}'

def make_test_fake_df(): 
    valid_df = pd.read_csv(f'{KAGGLE_DIR}/train.csv')
    valid_df.loc[:,'id']=valid_df['id'].astype(str) 
    fake_test_df=[]
    for i,d in valid_df.iterrows():
        #if i==4: break
        image_id = d['id']
    
        truth_df = pd.read_csv(f'{KAGGLE_DIR}/train/{image_id}/{image_id}.csv')
        non_nan_count = truth_df.count()
        #print(i,image_id,non_nan_count)
        #print(non_nan_count.index)
    
        #lead	fs	number_of_rows 
        this_df = pd.DataFrame({
            'id':image_id ,
            'lead':non_nan_count.index,
            'fs': d['fs'],
            'number_of_rows':non_nan_count.values 
        })
        fake_test_df.append(this_df)
        if i==0: print(this_df)
    fake_test_df = pd.concat(fake_test_df)
    return fake_test_df

In [ ]:
# set valid/test data
if MODE == 'local':
    valid_df = pd.read_csv(f'{KAGGLE_DIR}/train.csv')
    valid_df['id'] = valid_df['id'].astype(str)

    all_ids = valid_df['id'].unique().tolist()
    rng = np.random.default_rng(42)
    rng.shuffle(all_ids)

    N_VAL_IDS = 50
    val_image_ids = all_ids[:N_VAL_IDS]

    TYPE_IDS = ['0001', '0003', '0004', '0005', '0006', '0009', '0010', '0011', '0012']
    valid_id = [f'{image_id}-{type_id}' for image_id in val_image_ids for type_id in TYPE_IDS]



In [ ]:
    
if MODE == 'submit':
	valid_df = pd.read_csv(f'{KAGGLE_DIR}/test.csv')
	valid_df['id']=valid_df['id'].astype(str) 
	valid_id = valid_df['id'].unique().tolist()

if MODE == 'fake':
	valid_df = make_test_fake_df()
	valid_df['id']=valid_df['id'].astype(str) 
	valid_id = valid_df['id'].unique().tolist()

#--------------------------------------

def read_image(sample_id):
    if MODE == 'local':
        image_id, type_id = sample_id.split('-')
        image = cv2.imread(f'{KAGGLE_DIR}/train/{image_id}/{image_id}-{type_id}.png', cv2.IMREAD_COLOR_RGB)
        return image
    if MODE == 'submit':
        image_id = sample_id
        image = cv2.imread(f'{KAGGLE_DIR}/test/{image_id}.png', cv2.IMREAD_COLOR_RGB)
        return image
    if MODE == 'fake':
        image_id = sample_id 
        type_id = ['0001', '0003', '0004', '0005', '0006', '0009', '0010', '0011', '0012'][
            int(image_id)%9
        ] 
        image = cv2.imread(f'{KAGGLE_DIR}/train/{image_id}/{image_id}-{type_id}.png', cv2.IMREAD_COLOR_RGB)
        return image

def read_sampling_length(sample_id):
	if MODE == 'local':
		image_id, type_id = sample_id.split('-')
		d = valid_df[valid_df['id']==image_id].iloc[0]
		length = d.sig_len
		return length
	if MODE == 'submit':
		image_id = sample_id
		d = valid_df[
			(valid_df['id']==image_id) & (valid_df['lead']=='II')
		].iloc[0]
		length = d.number_of_rows
		return length
	if MODE == 'fake':
		image_id = sample_id
		d = valid_df[
			(valid_df['id']==image_id) & (valid_df['lead']=='II')
		].iloc[0]
		length = d.number_of_rows
		return length

#valid_id = valid_id[:300]
print('valid_id:', len(valid_id))
print('\t', valid_id[:3], '...')
print('setting ok!!!\n')

In [ ]:
print("MODE:", MODE)
print("valid_df shape:", valid_df.shape)
print("unique ids:", valid_df['id'].nunique())
print(valid_df.head(20))
print(valid_df['id'].value_counts())

In [ ]:
# stage0
print('*** STARTING STAGE0 ***')

from stage0_model import Net as Stage0Net
from stage0_common import *
#image_to_batch,output_to_predict,normalise_by_homography,draw_results_stage0

os.makedirs(f'{OUT_DIR}/normalised', exist_ok=True)

def run_stage0():
   # Load the pre-trained Stage 0 model
	stage0_net = Stage0Net(pretrained=False)
	stage0_net = load_net(stage0_net, f'{WEIGHT_DIR}/stage0-last.checkpoint.pth')
	stage0_net.to(DEVICE)
    # .to(DEVICE) moves all model parameters to GPU
    # Essential: if model is on CPU but data is on GPU → error
    # If model is on GPU but data is on CPU → error
    # They must be on the same device

	start_timer = timer()
	for n, sample_id in enumerate(valid_id):
		timestamp = time_to_str(timer() - start_timer, 'sec')
		print(f'\r\t {n:4d} {sample_id}', timestamp, end='', flush=True)

		image = read_image(sample_id)
        # image shape: (H, W, 3) — e.g., (3024, 4032, 3) for a phone photo
        # dtype: uint8, values 0-255
		batch = image_to_batch(image)
        # Converts raw image to model-ready tensor:
        # 1. Resize to fixed input size (e.g., 512×512 or 768×1024)
        # 2. Normalize: (pixel/255 - mean) / std  (ImageNet mean and std stats cause we will use res/efficentNet that is trained on thatits what it excpects)
        #eg:two channels are both 0.05 above their mean, that 0.05 should count as more important in a channel that usually varies very little than in a channel that naturally varies a lot. Dividing by std captures exactly that idea.
        # 3. Rearrange: HWC → CHW (channels first, required by PyTorch)
        # 4. Add batch dim: CHW → BCHW (B=1 for single image)
        # 5. Move to DEVICE
        # Result shape: (1, 3, H_model, W_model)

		with torch.amp.autocast('cuda', dtype=FLOAT_TYPE):
        # autocast: automatically uses float16 for most operations
        # → 2× less GPU memory, 1.5-2× faster
        # → slight precision loss, negligible for inference
			with torch.no_grad():
				output = stage0_net(batch)
        # no_grad(): tells PyTorch not to build computation graph
        # → saves memory (no gradient tensors allocated)
        # → faster (skip gradient tracking)
        # ALWAYS use no_grad() during inference

				try:
					rotated, keypoint = output_to_predict(image, batch, output)
            # rotated: image after rotation correction (H×W×3, uint8)
            # keypoint: detected corner keypoints of the ECG paper in the image (the four corners of the paper i think: no the 9 start of the middle leads actually thats whhy the homography will be 3x3 ig)
					normalised, keypoint, homo = normalise_by_homography(rotated, keypoint)
			# normalised: image after full homography correction (from the 9 keypoints)
            # homo: the 3×3 homography matrix H
            # Shape of normalised: always (1700, 2200, 3) — fixed canonical size
					cv2.imwrite(f'{OUT_DIR}/normalised/{sample_id}.norm.png', cv2.cvtColor(normalised, cv2.COLOR_RGB2BGR))
					np.save(f'{OUT_DIR}/normalised/{sample_id}.homo.npy', homo)
            # Save to disk: Stage 1 reads these files later
            # Saving homo.npy: needed if you want to reconstruct original coordinates
				except:
					FAIL_ID.append(sample_id)

		torch.cuda.empty_cache()
    # empty_cache(): releases GPU memory that PyTorch cached
    # Important after failures: a failed forward pass might leave
    # partial tensors on GPU → memory leak → OOM on next iteration
		#if n<10: # optional: show results
			#overlay = draw_results_stage0(rotated, keypoint)
            # So the overlay is based on the keypoint set after that normalization step has processed it and there still compatible so the new keypoints work with non normlized images to just rotated ?
			#print('')
			#print('demo results for stage0--------------')
			#print(sample_id)
			#plt.imshow(image);plt.show()
			#plt.imshow(overlay);plt.show()
			#plt.imshow(normalised);plt.show()
			
	print('')

run_stage0()
print('FAIL_ID:', FAIL_ID)
print('run_stage0() ok!!!\n')

### 4.3 Inside `image_to_batch` — what preprocessing actually happens

`image_to_batch(image)` does **not** directly apply ImageNet normalization, and it does **not** resize the image to a fixed shape such as `(512, 512)`. [file:144][file:145]  
Instead, it first reads the original image size `(H, W)`, computes a scale factor `scale = WIDTH / W` with `WIDTH = 1440`, and resizes the image while keeping the aspect ratio. [file:144]  
The resize uses `cv2.INTER_AREA`, which is a common choice for shrinking images. [file:144]  

After resizing, the function pads the image with zeros so that the padded height and width are slightly larger than the resized image and aligned to multiples of 32. [file:144]  
This is useful because the Stage 0 network is a convolutional encoder-decoder, and such architectures usually work more cleanly when spatial dimensions are compatible with repeated downsampling and upsampling. [file:145][file:144]  

Next, the function converts the image from HWC format to CHW format and adds a batch dimension, so one image becomes a tensor of shape `(1, 3, H, W)`. [file:144]  
Then it creates 4 test-time augmentation versions: the original image, a vertical flip, a horizontal flip, and a flip in both directions. [file:144]  
These 4 versions are concatenated into one batch, and the function also stores metadata such as the original size, resized size, and scale factor. [file:144]  

So the real role of `image_to_batch` is: resize with aspect ratio preserved, pad, convert to tensor layout, and build the TTA batch. [file:144]  
The actual ImageNet normalization happens later inside the model's `forward()` function, where the tensor is converted with `x = image.float() / 255` and then `x = (x - mean) / std`. [file:145]  

---

### 4.4 Inside `output_to_predict` — decoding the real Stage 0 outputs

The Stage 0 network is a two-head model. [file:145]  
One head predicts `marker`, which is a segmentation-like per-pixel class map, and the other head predicts `orientation`, which is an image-level rotation class. [file:145]  
So Stage 0 does **not** predict the 4 paper corners directly. [file:145][file:144]  

More precisely, the model defines `self.marker = nn.Conv2d(..., 13 + 1, kernel_size=1)` and `self.orientation = nn.Linear(..., 8)`. [file:145]  
During inference, both outputs are passed through `softmax`, so `marker` becomes per-pixel class probabilities and `orientation` becomes class probabilities over 8 rotation-related classes. [file:145]  

Inside `output_to_predict`, the code loops over the 4 TTA images. [file:144]  
For each flipped input, it undoes the flip on the predicted `marker` map and also reorders the `orientation` probabilities so they match the original image reference frame before averaging. [file:144]  
After that, it averages the 4 predictions into one final `marker` map and one final `orientation` vector. [file:144]  

Then the function calls `marker_to_keypoint(image, orientation, marker, scale)`. [file:144]  
Inside `marker_to_keypoint`, the first step is `k = orientation.argmax()`, which means the image rotation is chosen **directly from the orientation head**, not computed later from the keypoints. [file:144][file:145]  
The predicted marker map is then rotated with `np.rot90(...)` so that landmark extraction happens in the predicted upright orientation. [file:144]  

Next, the code converts the soft marker probabilities into a hard label image with `thresh = marker.argmax(-1)`. [file:144]  
For each of the 9 labels `[2, 3, 4, 6, 7, 8, 10, 11, 12]`, it finds connected components with `cc3d.connected_components(thresh == label)`, computes the component statistics, sorts the components by size, and keeps the centroid of the largest one. [file:144]  
That centroid becomes the final landmark coordinate for that class. [file:144]  

So the model does **not** directly output 9 `(x, y)` coordinates either. [file:144][file:145]  
Instead, it outputs a marker map, and the code converts that map into 9 keypoints by taking one centroid per landmark class. [file:144]  

The 9 landmarks correspond to the lead-start positions for `aVR`, `V1`, `V4`, `aVL`, `V2`, `V5`, `aVF`, `V3`, and `V6`. [file:144]  
Therefore, the Stage 0 keypoints are internal ECG-layout landmarks, not the four corners of the paper. [file:144]  

Finally, `output_to_predict` rotates the original image with the chosen rotation `k` and returns `(rotated, keypoint)`. [file:144]  
At this moment, each keypoint has the form `[x, y, label, leadname]`. [file:144]  

A useful intuition is this: the marker head paints small class-specific blobs near important ECG landmarks, and `cc3d` turns each blob into one clean center point. [file:144][file:145]  

---

### 4.5 Inside `normalise_by_homography` — the real geometric normalization

`normalise_by_homography(image, keypoint)` does **not** use 4 paper corners. [file:144]  
Instead, it extracts the 9 detected landmark coordinates with `pt9 = [[k[0], k[1]] for k in keypoint]` and sends them to `normalise_image(image, pt9)`. [file:144]  

Inside `normalise_image`, the code computes a homography with `cv2.findHomography(pt9, ref_pt9, method=cv2.RANSAC)`. [file:144]  
So the source points are the 9 detected landmarks in the current rotated ECG image, and the destination points are 9 canonical reference points `REF_PT9`. [file:144]  
These reference points are built beforehand from `640106434-0001.gridpoint_xy.npy`, then scaled and shifted into the Stage 0 canonical frame. [file:144]  

This means the Stage 0 homography maps the detected lead-start landmarks in the current image to their standard template locations. [file:144]  
In other words, Stage 0 normalizes the ECG image by aligning its internal landmark layout to a canonical ECG layout. [file:144]  

The call uses `method=cv2.RANSAC`, which is important because not every detected keypoint is guaranteed to be perfect. [file:144]  
RANSAC tries to find one homography that is supported by the most geometrically consistent correspondences, while rejecting inconsistent ones as outliers. [file:144]  
This is safer than forcing all 9 points to agree exactly when one or more detections may be slightly wrong. [file:144]  

The function then warps the image with `cv2.warpPerspective(image, homo, (WIDTH, HEIGHT))`, where `WIDTH = 1440` and `HEIGHT = 1152`. [file:144]  
So the Stage 0 normalized image has shape `(1152, 1440, 3)`, not `(1700, 2200, 3)`. [file:144][file:95]  
The larger canonical geometry such as `1700 x 2200` appears elsewhere in the pipeline context, but the actual Stage 0 normalization code in `stage0_common.py` outputs `(1152, 1440)`. [file:144][file:95]  

`cv2.findHomography` also returns a `match` mask. [file:144]  
This mask tells which of the 9 correspondences were considered inliers to the final homography estimated by RANSAC. [file:144]  
`normalise_by_homography` appends this flag to each keypoint, so each keypoint changes from `[x, y, label, leadname]` to `[x, y, label, leadname, match]`. [file:144]  

This is why `draw_results_stage0(rotated, keypoint)` still works correctly on the rotated image. [file:144]  
The keypoint coordinates are still in the rotated-image coordinate system, and the only new information added is whether each point was an inlier or outlier for the homography fit. [file:144]  
In the visualization, matched points are drawn as filled circles with the lead name written next to them, while unmatched points are drawn as outlined circles. [file:144]  

So the correct Stage 0 interpretation is: predict orientation, predict a marker map, extract 9 lead-start landmarks with connected components, fit a homography from those 9 detected points to 9 canonical template points using RANSAC, and warp the image into a standard Stage 0 normalized frame. [file:144][file:145]  

Why heatmaps instead of directly regressing (x,y) coordinates?
-

This is a fundamental architectural choice that appears in many pose estimation problems (human body keypoints, face landmarks, etc.).

Direct regression approach:

text
CNN → flatten → dense layer → [x, y]  (2 numbers)
Problem: the network must compress all spatial information into 2 numbers through a bottleneck. If the corner is in the top-right of the image, the network must somehow encode "top-right" as a single number. This is hard to learn because there's no spatial structure in the output.

Heatmap approach:

CNN → UNet decoder → heatmap (H_feat × W_feat)  (spatial output)
The network outputs a 2D map where the peak is at the corner location. The spatial structure of the output directly mirrors the spatial structure of the input. The network just needs to learn "draw a peak where you see a corner mark" — much more natural for a convolutional architecture.

Why does this work so well with CNNs?
Convolutions are inherently spatial operations. Each spatial position in the feature map "knows" what's happening at the corresponding location in the input image (within its receptive field). Outputting a heatmap leverages this spatial awareness directly. Outputting a single (x,y) via a dense layer destroys this spatial structure.

 The heatmap approach preserves the topological structure of the problem — nearby pixels in the input correspond to nearby outputs. Direct regression via dense layers breaks this topology completely (all spatial information is mixed in the flattened vector).

-

 

Why type 0001 images for training Stage 1?
-
Type 0001 is the synthetic original — perfect image, no noise, no distortion, no color cast. The gridlines are perfectly sharp and perfectly positioned. This gives pixel-perfect ground truth. The network then generalizes to other types (3, 4, 5, etc.) because after Stage 0 normalisation, all types look somewhat similar (roughly aligned, similar scale). The remaining differences (noise, blur, color) don't prevent gridline detection — the network learned to find gridlines in clean images and generalizes well enough to noisy ones.

Understanding the homography matrix H mathematically:

A homography maps points between two projective planes. In homogeneous coordinates:

text
[x']     [h00  h01  h02]   [x]
[y']  =  [h10  h11  h12] × [y]
[w']     [h20  h21  h22]   [1]

Then: x_canonical = x'/w',  y_canonical = y'/w'
The division by w' is what makes it a projective (not just affine) transformation. When w'=1 everywhere, it reduces to an affine transformation. When w' varies spatially, you get perspective effects — lines remain straight but parallel lines can converge (like railroad tracks vanishing to a point).

Why exactly 4 points to determine H?

H has 9 elements but only 8 degrees of freedom (the overall scale doesn't matter — you can multiply all elements by any constant). Each point correspondence gives 2 equations (x and y). So 4 points × 2 equations = 8 equations → exactly determines all 8 degrees of freedom. With fewer points the system is underdetermined. With more points you'd use RANSAC (as in Stage 1).

cv2.warpPerspective — what it actually does pixel by pixel:

text
For each output pixel (x_out, y_out) in canonical space:
    1. Compute source position: [x_src, y_src, w] = H_inv @ [x_out, y_out, 1]
    2. Normalize: x_src /= w,  y_src /= w
    3. Sample the rotated image at (x_src, y_src) using bilinear interpolation
    4. Place that color at output position (x_out, y_out)
This is called inverse warping (you go from output to input to find what to sample). The alternative, forward warping (map each input pixel to its output position), creates holes (multiple input pixels might map to the same output, or some outputs get nothing). Inverse warping fills every output pixel exactly once — no holes, no double-filling.

Bilinear interpolation at step 3:
The source position is usually non-integer (e.g., 347.3, 892.7). Bilinear interpolation blends the 4 surrounding pixels using weights proportional to the fractional distances:

text
pixel(347.3, 892.7) =
    0.7 × 0.3 × pixel(347, 892) +   (weight = (1-0.3)×(1-0.7))
    0.7 × 0.7 × pixel(347, 893) +   (weight = (1-0.3)×0.7)
    0.3 × 0.3 × pixel(348, 892) +   (weight = 0.3×(1-0.7))
    0.3 × 0.7 × pixel(348, 893)     (weight = 0.3×0.7)
This gives smooth, anti-aliased results. Compare with nearest-neighbor interpolation (just round to nearest integer pixel) which produces blocky/jagged edges.

In [ ]:
# stage1
print('*** STARTING STAGE1 ***')

from stage1_model import Net as Stage1Net
from stage1_common import *
# output_to_predict (different from stage0's version) , rectify_image,draw_mapping,draw_results_stage1
os.makedirs(f'{OUT_DIR}/rectified', exist_ok=True)

In [ ]:
#gridpoint sparse mask improvment from solution 1 : THE GRIDPOINTS WEIGHTED AVERGE THAT HAS SUB PIXEL PRESISION ( after the model output we just dont use cc3d alone we averge around the confident detections ) THE SURFACE FITTING POLYNOMIALE( so the detetected points make sens together more)

def _poly_design(ii, jj):
    ii = np.asarray(ii, np.float32)
    jj = np.asarray(jj, np.float32)
    return np.stack([
        np.ones_like(ii),
        ii,
        jj,
        ii * ii,
        jj * jj,
        ii * jj,
    ], axis=1)
# For each detected blob, it computes a weighted average of all pixel positions inside it, using the heatmap probability value at each pixel as the weight.

def weighted_cc_centroids(prob, thresh=0.5, min_area=3, eps=1e-8):
    cc = cc3d.connected_components(prob > thresh)
    pts = []
    for k in range(1, cc.max() + 1):
        ys, xs = np.where(cc == k)
        if len(xs) < min_area:
            continue

        w = prob[ys, xs].astype(np.float64)
        s = w.sum()
        if s < eps:
            continue

        y = (ys * w).sum() / (s + eps)
        x = (xs * w).sum() / (s + eps)
        pts.append([y, x])

    if len(pts) == 0:
        return np.zeros((0, 2), np.float32)

    return np.asarray(pts, np.float32)

def subpixel_point_to_label(y, x, hcc, vcc):
    H, W = hcc.shape

    y0 = int(np.clip(np.floor(y), 0, H - 1))
    x0 = int(np.clip(np.floor(x), 0, W - 1))
    y1 = min(y0 + 1, H - 1)
    x1 = min(x0 + 1, W - 1)

    candidates = [
        (hcc[y0, x0], vcc[y0, x0]),
        (hcc[y0, x1], vcc[y0, x1]),
        (hcc[y1, x0], vcc[y1, x0]),
        (hcc[y1, x1], vcc[y1, x1]),
    ]
    candidates = [(j, i) for j, i in candidates if j > 0 and i > 0]

    if len(candidates) == 0:
        return 0, 0

    vals, counts = np.unique(np.asarray(candidates), axis=0, return_counts=True)
    j, i = vals[counts.argmax()]
    return int(j), int(i)

def refine_grid_surface(gridpoint_xy, sigma=3.0, fill_missing=True):
    out = gridpoint_xy.copy()

    valid = np.any(out != 0, axis=2)
    jj, ii = np.where(valid)  # row, col on canonical grid

    if len(ii) < 12:
        return out

    A = _poly_design(ii, jj).astype(np.float64)
    tx = out[jj, ii, 0].astype(np.float64)
    ty = out[jj, ii, 1].astype(np.float64)

    cx, *_ = np.linalg.lstsq(A, tx, rcond=None)
    cy, *_ = np.linalg.lstsq(A, ty, rcond=None)

    px = A @ cx
    py = A @ cy

    resid = np.sqrt((tx - px) ** 2 + (ty - py) ** 2)
    thr = resid.mean() + sigma * (resid.std() + 1e-6)
    bad = resid > thr

    if bad.any():
        out[jj[bad], ii[bad], 0] = px[bad].astype(np.float32)
        out[jj[bad], ii[bad], 1] = py[bad].astype(np.float32)

    if fill_missing:
        jg, ig = np.meshgrid(
            np.arange(out.shape[0], dtype=np.float32),
            np.arange(out.shape[1], dtype=np.float32),
            indexing='ij'
        )
        Afull = _poly_design(ig.reshape(-1), jg.reshape(-1)).astype(np.float64)
        fullx = (Afull @ cx).reshape(out.shape[:2]).astype(np.float32)
        fully = (Afull @ cy).reshape(out.shape[:2]).astype(np.float32)

        miss = ~valid
        out[..., 0][miss] = fullx[miss]
        out[..., 1][miss] = fully[miss]

    return out

def output_to_predict_refined(image, batch, output, interpolate_mapping_fn,
                              segment_to_endpoints_fitline_fn,
                              canonical_x_order_fn,
                              canonical_y_order_fn,
                              compare_segment_fn):
    marker = output['marker'][0]
    gridpoint = output['gridpoint'][0, 0]
    gridhline = output['gridhline'][0]
    gridvline = output['gridvline'][0]

    marker = marker.argmax(0).byte().data.cpu().numpy()
    gridpoint = gridpoint.float().data.cpu().numpy()
    gridhline = gridhline.float().data.cpu().numpy()
    gridvline = gridvline.float().data.cpu().numpy()

    gridvline = gridvline.argmax(0).astype(np.uint8)
    gridhline = gridhline.argmax(0).astype(np.uint8)

    # 1) sub-pixel point locations: weighted centroid instead of plain CC centroid
    point_yx = weighted_cc_centroids(gridpoint, thresh=0.5, min_area=3)

    # 2) same line filtering logic as baseline
    gvfiltered = np.zeros_like(gridvline)
    cc = cc3d.connected_components(gridvline != 0)
    num_line = cc.max()
    for l in range(1, num_line + 1):
        t = (cc == l)
        bincount = np.bincount(gridvline[t])
        c = bincount.argmax()
        gvfiltered[t] = c

    gvreject = np.zeros_like(gridvline)
    for l in range(1, num_line + 1):
        cc2 = cc3d.connected_components(gvfiltered == l)
        if cc2.max() > 1:
            num = cc2.max() + 1
            stats = cc3d.statistics(cc2)
            area = stats['voxel_counts'][1:]
            label = np.arange(1, num)

            argsort = np.argsort(area)[::-1]
            area = area[argsort]
            label = label[argsort]

            if area[0] < 7:
                continue

            main_segment = segment_to_endpoints_fitline_fn(cc2 == label[0])
            main_segment = canonical_y_order_fn(*main_segment)

            for j in range(1, len(label)):
                if area[j] < 7:
                    continue
                segment = segment_to_endpoints_fitline_fn(cc2 == label[j])
                segment = canonical_y_order_fn(*segment)
                ang_dis, ori_dis, seg_dis = compare_segment_fn(main_segment, segment)

                if ori_dis > 5:
                    gvreject[cc2 == label[j]] = 255
                else:
                    gvfiltered[cc2 == label[j]] = l

    vcc = gvfiltered.copy()

    ghfiltered = np.zeros_like(gridhline)
    cc = cc3d.connected_components(gridhline != 0)
    num_line = cc.max()
    for l in range(1, num_line + 1):
        t = (cc == l)
        bincount = np.bincount(gridhline[t])
        c = bincount.argmax()
        ghfiltered[t] = c

    ghreject = np.zeros_like(gridhline)
    for l in range(1, num_line + 1):
        cc2 = cc3d.connected_components(ghfiltered == l)
        if cc2.max() > 1:
            num = cc2.max() + 1
            stats = cc3d.statistics(cc2)
            area = stats['voxel_counts'][1:]
            label = np.arange(1, num)

            argsort = np.argsort(area)[::-1]
            area = area[argsort]
            label = label[argsort]

            if area[0] < 7:
                continue

            main_segment = segment_to_endpoints_fitline_fn(cc2 == label[0])
            main_segment = canonical_x_order_fn(*main_segment)

            for j in range(1, len(label)):
                if area[j] < 7:
                    continue
                segment = segment_to_endpoints_fitline_fn(cc2 == label[j])
                segment = canonical_x_order_fn(*segment)
                ang_dis, ori_dis, seg_dis = compare_segment_fn(main_segment, segment)

                if ori_dis > 5:
                    ghreject[cc2 == label[j]] = 255
                else:
                    ghfiltered[cc2 == label[j]] = l

    hcc = ghfiltered.copy()

    # 3) build sparse (44,57,2) grid using sub-pixel points
    gridpoint_xy = np.zeros((44, 57, 2), np.float32)

    for y, x in point_yx:
        j, i = subpixel_point_to_label(y, x, hcc, vcc)
        if (j == 0) or (i == 0):
            continue

        jj = j - 1
        ii = i - 1

        if np.all(gridpoint_xy[jj, ii] == 0):
            gridpoint_xy[jj, ii] = [x, y]
        else:
            oy, ox = int(round(gridpoint_xy[jj, ii, 1])), int(round(gridpoint_xy[jj, ii, 0]))
            ny, nx = int(round(y)), int(round(x))

            oy = np.clip(oy, 0, gridpoint.shape[0] - 1)
            ox = np.clip(ox, 0, gridpoint.shape[1] - 1)
            ny = np.clip(ny, 0, gridpoint.shape[0] - 1)
            nx = np.clip(nx, 0, gridpoint.shape[1] - 1)

            old_score = gridpoint[oy, ox]
            new_score = gridpoint[ny, nx]
            if new_score > old_score:
                gridpoint_xy[jj, ii] = [x, y]

    # 4) 1st-place-style surface fitting: replace outliers on the sparse lattice
    gridpoint_xy = refine_grid_surface(gridpoint_xy, sigma=3.0, fill_missing=False)

    # 5) keep baseline hole filling
    gridpoint_xy = interpolate_mapping_fn(gridpoint_xy)

    more = {
        'ghfiltered': ghfiltered,
        'gvfiltered': gvfiltered,
    }
    return gridpoint_xy, more

| Step                    | Action                                                        | Why                                                 |
| ----------------------- | ------------------------------------------------------------- | --------------------------------------------------- |
| weighted_cc_centroids   | Get sub-pixel (y, x) for each detected grid point blob        | Reduces initial quantization error                  |
| subpixel_point_to_label | Assign each point to its (row, col) in the 44×57 grid         | Maps detections to canonical grid positions         |
| refine_grid_surface     | Fit polynomial, flag 3σ outliers, replace with surface values | Removes bad grid points that would warp incorrectly |
| interpolate_mapping_fn  | Fill any remaining holes                                      | Inherited from the baseline                         |

In [ ]:
# 
def run_stage1():
    stage1_net = Stage1Net(pretrained=False)
    stage1_net = load_net(stage1_net, f'{WEIGHT_DIR}/stage1-last.checkpoint.pth')
    stage1_net.to(DEVICE)

    start_timer = timer()
    for n, sample_id in enumerate(valid_id):
        timestamp = time_to_str(timer() - start_timer, 'sec')
        print(f'\r\t {n:4d} {sample_id}', timestamp, end='', flush=True)

        if sample_id in FAIL_ID:
            continue

        image = cv2.imread(f'{OUT_DIR}/normalised/{sample_id}.norm.png', cv2.IMREAD_COLOR_RGB)

        batch = {
            'image': torch.from_numpy(
                np.ascontiguousarray(image.transpose(2, 0, 1))
            ).unsqueeze(0),
        }

        num_tta = 1

        with torch.amp.autocast('cuda', dtype=FLOAT_TYPE):
            with torch.no_grad():
                output = stage1_net(batch)

        try:
            gridpoint_xy, more = output_to_predict_refined(
                image=image,
                batch=batch,
                output=output,
                interpolate_mapping_fn=interpolate_mapping,
                segment_to_endpoints_fitline_fn=segment_to_endpoints_fitline,
                canonical_x_order_fn=canonical_x_order,
                canonical_y_order_fn=canonical_y_order,
                compare_segment_fn=compare_segment,
            )
           # gridpoint_xy, more = output_to_predict(image, batch, output)
            rectified = rectify_image(image, gridpoint_xy)

            cv2.imwrite(
                f'{OUT_DIR}/rectified/{sample_id}.rect.png',
                cv2.cvtColor(rectified, cv2.COLOR_RGB2BGR)
            )
            np.save(f'{OUT_DIR}/rectified/{sample_id}.gridpoint_xy.npy', gridpoint_xy)

        except Exception as e:
            print(f'\nStage1 failed for {sample_id}: {e}')
            FAIL_ID.append(sample_id)

        torch.cuda.empty_cache()

       # if n < 10:
           # overlay = draw_mapping(image, gridpoint_xy)
           # ghfiltered, gvfiltered = draw_results_stage1(more)

          #  print('')
           # print('demo results for stage1--------------')
            #print(sample_id)
           # plt.imshow(overlay)
           # plt.show()
           # plt.imshow(gvfiltered)
           # plt.show()
           # plt.imshow(ghfiltered)
            #plt.show()
           # plt.imshow(rectified)
           # plt.show()

    print('')


run_stage1()
print('FAIL_ID:', FAIL_ID)
print('run_stage1() ok!!!\n')

### 5.3 The Stage 1 Neural Network Architecture

Stage 1 takes as input the **Stage 0 normalized image**, whose size is fixed at `(1152, 1440, 3)`. [file:144][file:146][file:147]  
The model then applies ImageNet-style normalization inside its `forward()` function with `x = image.float() / 255` followed by `(x - mean) / std`. [file:147]  

Stage 1 is not just a "horizontal heatmap + vertical heatmap" model. [file:147][file:146]  
It has **four output heads**: `gridpoint` with 1 channel, `gridhline` with `44+1=45` channels, `gridvline` with `57+1=58` channels, and `marker` with `13+1=14` channels. [file:147]  
So the network predicts candidate grid intersections, a per-pixel horizontal-line class, a per-pixel vertical-line class, and also a marker map. [file:147]  

The backbone is `resnet34.a3_in1k`, created with `num_classes=0` and `global_pool=''`, so the classifier head is removed and spatial feature maps are preserved. [file:147]  
The encoder feature dimensions are `[64, 128, 256, 512]`, and the decoder dimensions are `[256, 128, 64, 32]`. [file:147]  
The decoder is `MyUnetDecoder`, which is a standard UNet-style decoder with interpolation, skip connections, and convolution blocks; there is no coordinate-convolution module in this code. [file:147]  

More concretely, the decoder upsamples the deepest encoder feature map in four stages and combines it with skip features from earlier encoder levels. [file:147]  
The final decoder output has 32 channels, and the four prediction heads are applied with `1x1` convolutions on top of that shared feature map. [file:147]  
This means Stage 1 is a **multi-head dense prediction model**, not a model with a single 4-channel "pixel" output. [file:147]  

At inference time, `output_to_predict` converts `gridpoint` into a binary candidate-point map by thresholding `gridpoint > 0.5`, then extracts connected components and uses their centroids as candidate intersection locations. [file:146]  
For `gridhline` and `gridvline`, it does `argmax` over channels, so these are not simple heatmaps of "line present or absent"; they are pixelwise class maps that try to assign each pixel to one of the 44 horizontal lines or 57 vertical lines, plus background. [file:146][file:147]  
The code then cleans and links line segments using connected components, line fitting, and geometric consistency checks before keeping filtered horizontal and vertical line-label maps. [file:146]  

After that, each candidate point is assigned a horizontal line index `j` and a vertical line index `i` by reading the filtered maps at that point location. [file:146]  
The result is stored in `gridpoint_xy` with shape `(44, 57, 2)`, where `gridpoint_xy[j-1, i-1] = [x, y]` gives the image-space coordinate of that canonical grid intersection. [file:146]  
Missing entries are then filled by `interpolate_mapping(gridpoint_xy)` using cubic interpolation with `scipy.interpolate.griddata`. [file:146]  

So the main goal of Stage 1 is not merely to detect isolated intersections. [file:146][file:147]  
Its real goal is to recover a **dense mapping** from the canonical ECG grid to the current image, using point candidates together with horizontal and vertical line identities. [file:146]  
That mapping is exactly what makes precise geometric rectification possible. [file:146]  

Finally, `rectify_image(image, gridpoint_xy)` converts the sparse `(44, 57, 2)` mapping into a dense sampling map, upsamples it to a reference size `(1700, 2200)`, and uses `torch.nn.functional.grid_sample` to warp the image. [file:146]  
So the Stage 1 **output image** is rectified to `(1700, 2200)`, but the Stage 1 **input image** is still the Stage 0 normalized image of size `(1152, 1440)`. [file:146][file:147]  
This is why saying "Stage 1 takes a 1700x2200 image" is incorrect; 1700x2200 is the rectified output size, not the network input size. [file:146][file:147]
the training target used is the 001 scaled and added output to each cordinate to match the stage 0 output and stage 1 working space , when we do it for most photos the coordinantes of grid and all are veeery colse only rounding probs maybe , but for the mobile photos if its not perfectly centerd in the corners we may not have perfect training target but we would have moset good cause we did a homography rectification before in stage0 and the ones that are bad and not guussed even in the corners would be calculated by the interpolation to reach the number cause as long as we have a good number of grid pointes detecterd the rest can be deduced by cubic interpolating i guess . for the 001 we got every infos there iso. the transformation is 280 / 2200 and add (-6, +10) for each x,y in canonical . 
imprvment till now to add : per type preprocess more tta in stage 1 and subpixel targets in stage 1 

Why type 0001 images for training Stage 1?
-
Type 0001 is the synthetic original — perfect image, no noise, no distortion, no color cast. The gridlines are perfectly sharp and perfectly positioned. This gives pixel-perfect ground truth. The network then generalizes to other types (3, 4, 5, etc.) because after Stage 0 normalisation, all types look somewhat similar (roughly aligned, similar scale). The remaining differences (noise, blur, color) don't prevent gridline detection — the network learned to find gridlines in clean images and generalizes well enough to noisy ones.

cc3d role :
-
 After sigmoid: gh is a (850, 1100) float map with values in [0,1]
 Threshold to binary
gh_binary = (gh_raw > 0.5).astype(np.uint8)

 Label connected components
labels = cc3d.connected_components(gh_binary, connectivity=4)
 Each connected blob gets a unique integer label
 A genuine horizontal gridline → long, thin horizontal blob
 Noise / false positives → small, compact blobs

 Filter: keep only blobs with width >> height (horizontal lines)
for label_id in np.unique(labels):
    blob = (labels == label_id)
    blob_width  = blob.any(axis=0).sum()   # number of columns the blob spans
    blob_height = blob.any(axis=1).sum()   # number of rows the blob spans
    if blob_width < 50 or blob_height > 20:
        gh_binary[blob] = 0   # remove non-line blobs
 Result: only genuine horizontal lines remain
 Why connected components instead of just thresholding?
Simple thresholding removes weak detections. Connected component analysis removes geometrically wrong detections — even confident ones. A false positive with activation 0.8 but spanning only 10 columns (not a full-width line) gets removed. A genuine gridline spanning 800 columns gets kept. Geometry matters more than raw confidence.

Why cc3d instead of scipy.ndimage.label?
cc3d is dramatically faster for large binary arrays (1700×2200). It uses a highly optimized C++ implementation with parallel processing. For the Stage 1 heatmaps at half resolution (850×1100), scipy.ndimage.label might take several seconds. cc3d does it in milliseconds. In a competition with runtime limits, this difference matters across thousands of test images.

RMQ: The gridpointxy.npy File — Why It's Saved
This saved file is used by multiple downstream solutions:

hengck23 Stage 2: Reads the file to know the zero_mv pixel positions (where the ECG baseline is in the rectified image) and the time span t0, t1 (which columns of the rectified image correspond to the start and end of the ECG signal).

2nd place solution: Reads gridpointxy.npy directly to perform their own refined warping (they trust hengck23's grid detection but apply their own coordinate transformations on top, mapping from the competition's resampled signals to PTB-XL's 500Hz original).

In [ ]:
# stage2
print('*** STARTING STAGE2 ***')

import torch.nn as nn
from stage2_smp_model import Net as WholeModel
from stage2_lead_model import Net as LeadModel
from stage2_model import prob_to_series_by_max   
from stage2_common import *

os.makedirs(f'{OUT_DIR}/digitalised', exist_ok=True)
#os.makedirs(f'{OUT_DIR}/debug', exist_ok=True)

In [ ]:
#global constants:
from scipy import signal   # add to top imports if not present

WINDOW_SIZE  = 240
OFFSET       = 416
IGNORE_EDGE  = 8
x_scale      = 5000 / (2080 - 118)
add_x        = 1
y_scale      = 1
IMG_H, IMG_W = int(1700 * y_scale), int(2200 * x_scale) + add_x

tta = [0]   # test-time augmentation: original + horizontal flip

x0, x1 = 0, 5600
y0, y1 = 0, 1696
zero_mv = [703.5, 987.5, 1271.5, 1531.5]        # pixel row of 0mV per row
zero_mv_trimed  = [pos - OFFSET for pos in zero_mv]
zero_mv_croped  = [WINDOW_SIZE + 0.5 for _ in range(4)]
mv_to_pixel     = 79.0
t0, t1 = int(118 * x_scale) + add_x, int(2080 * x_scale) + add_x

# Ensemble regions: for each lead row, which part of the trimmed image
# and which part of the cropped lead image to fuse
height_after_trimed = y1 - OFFSET
ens_regions = []
for zmv in zero_mv_trimed:
    trim_upper = int(zmv) - WINDOW_SIZE
    trim_lower = int(zmv) + WINDOW_SIZE
    lead_upper = IGNORE_EDGE
    lead_lower = -IGNORE_EDGE
    if trim_lower > height_after_trimed:
        lead_lower = (trim_lower - height_after_trimed + IGNORE_EDGE) * -1
        trim_lower = height_after_trimed
    trim_upper += IGNORE_EDGE
    trim_lower -= IGNORE_EDGE
    ens_regions.append([trim_upper, trim_lower, lead_upper, lead_lower])
#helper fonction : pixel_to_series_exp — this is the subpixel conversion. It replaces hengck23's pixel_to_series (which used argmax):
def pixel_to_series_exp(pixel, zero_mv, length):
    """Subpixel-precision pixel→series via weighted row expectation."""
    _, H, W = pixel.shape
    eps = 1e-8
    y_idx = np.arange(H, dtype=np.float32)[:, None]   # (H,1)
    series = []
    for j in [0, 1, 2, 3]:
        p = pixel[j]
        denom = p.sum(axis=0)           # (W,)
        y_exp = (p * y_idx).sum(axis=0) / (denom + eps)   # soft expectation
        series.append(y_exp)
    series = np.stack(series).astype(np.float32)
    if length is not None and length != W:
        series = np.stack([
            signal.resample(s, length).astype(np.float32) for s in series
        ])
    return series
#read_images — loads the rectified image and builds the 4 cropped lead images for LeadModel:
def read_images(path):
    image = cv2.imread(path, cv2.IMREAD_COLOR)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (IMG_W, IMG_H), interpolation=cv2.INTER_LINEAR)
    trim_image = image.copy()[OFFSET:y1, x0:x1]

    image = image[y0:y1, x0:x1]
    H, W, _ = image.shape
    lead_images = []
    for i, zmv in enumerate(zero_mv):
        h0, h1 = int(zmv) - WINDOW_SIZE, int(zmv) + WINDOW_SIZE
        src_h0, src_h1 = max(0, h0), min(H, h1)
        dst_h0 = src_h0 - h0
        dst_h1 = dst_h0 + (src_h1 - src_h0)
        lead_img = np.zeros((WINDOW_SIZE * 2, W, 3))
        lead_img[dst_h0:dst_h1, :, :] = image[src_h0:src_h1, :, :]
        lead_images.append(lead_img)
    lead_images = np.stack(lead_images)   # (4, H, W, 3)
    return trim_image, lead_images    
#loader fonctions    
def get_whole_model(encoder_name, weight_path, device):
    model = WholeModel(encoder_name=encoder_name, encoder_weights=None,
                       decoder_name="unet", use_coord_conv=True, pretrained=False)
    state_dict = torch.load(weight_path, map_location=lambda s, l: s)
    print(model.load_state_dict(state_dict, strict=False))
    model.to(device); model.eval(); model.output_type = ['infer']
    return model

def get_lead_model(encoder_name, weight_path, fusion_type, device):
    model = LeadModel(encoder_name=encoder_name, encoder_weights=None,
                      fusion_type=fusion_type)
    state_dict = torch.load(weight_path, map_location=lambda s, l: s)
    print(model.load_state_dict(state_dict, strict=False))
    model.to(device); model.eval(); model.output_type = ['infer']
    return model
def get_attention_lead_model(weight_path, device):
    model = LeadModel(
        encoder_name='tu-efficientnet_b6',
        encoder_weights=None,
        fusion_type='cross_attn',
        fusion_levels=[3, 4],
    )
    state = torch.load(weight_path, map_location='cpu')
    msg = model.load_state_dict(state, strict=False)
    print(msg)
    model.to(device)
    model.eval()
    model.output_type = ['infer']
    return model

In [ ]:
def run_stage2(gpu_id=0, assigned_ids=None, prev_fail_ids=None, fail_id_file=None):
    device = f'cuda:{gpu_id}'
    if assigned_ids is None: assigned_ids = valid_id
    if prev_fail_ids is None: prev_fail_ids = []
    local_fail_id = []

    # Load all models once
    whole_models = [
        get_whole_model("tu-timm/tf_efficientnet_b7.ns_jft_in1k",
            "/kaggle/input/datasets/takashisomeya/physionet-final-submission-models/whole_b7_lb22.93.pth", device),
        get_whole_model("tu-timm/tf_efficientnetv2_l.in21k",
            "/kaggle/input/datasets/takashisomeya/physionet-final-submission-models/whole_v2_l_lb22.60.pth", device),
    ]
    lead_models = [
        get_lead_model("tu-timm/tf_efficientnet_b6.ns_jft_in1k",
            "/kaggle/input/datasets/takashisomeya/physionet-final-submission-models/series_b6_shared_conv2d_lb23.10.pth", "shared_conv2d", device),
        get_lead_model("tu-timm/tf_efficientnet_b6.ns_jft_in1k",
            "/kaggle/input/datasets/takashisomeya/physionet-final-submission-models/series_b6_shared_conv2d_lb23.00.pth", "shared_conv2d", device),
        get_lead_model("tu-timm/tf_efficientnetv2_l.in21k",
            "/kaggle/input/datasets/takashisomeya/physionet-final-submission-models/series_v2_l_conv3d_lb22.92.pth", "conv3d", device),
        get_lead_model("tu-timm/tf_efficientnetv2_l.in21k",
            "/kaggle/input/datasets/takashisomeya/physionet-final-submission-models/series_v2_l_conv2d_lb22.85.pth", "conv2d", device),
        get_attention_lead_model("/kaggle/input/datasets/zahouaniyacine/attention-fusion-model-ecg/cross_attn_b6_final_1.pth", device),
    ]

    start_timer = timer()
    for n, sample_id in enumerate(assigned_ids):
        timestamp = time_to_str(timer() - start_timer, 'sec')
        print(f'\r\t {n:4d} {sample_id}', timestamp, end='', flush=True)
        if sample_id in prev_fail_ids: continue

        length = read_sampling_length(sample_id)
        trim_image, lead_images = read_images(f'{OUT_DIR}/rectified/{sample_id}.rect.png')

        pixel_ens = np.zeros((4, trim_image.shape[0], trim_image.shape[1]))

        # --- Whole model inference (full trimmed image) ---
        batch = {'image': torch.from_numpy(
            np.ascontiguousarray(trim_image.transpose(2,0,1))).unsqueeze(0)}
        batch_tta = {'image': torch.from_numpy(
            np.ascontiguousarray(np.fliplr(trim_image).copy().transpose(2,0,1))).unsqueeze(0)}
        with torch.amp.autocast('cuda', dtype=FLOAT_TYPE):
            with torch.no_grad():
                for model in whole_models:
                    for flip in tta:
                        if flip:
                            pixel = model(batch_tta)['pixel'].float().cpu().numpy()[0]
                            pixel = np.flip(pixel, axis=flip)
                        else:
                            pixel = model(batch)['pixel'].float().cpu().numpy()[0]
                        pixel_ens += pixel

        # --- Lead model inference (per-lead cropped windows) ---
        lead_t = torch.from_numpy(lead_images.transpose(0,3,1,2)).contiguous()
        batch = {'image': lead_t.unsqueeze(0)}
        batch_tta = {'image': torch.flip(lead_t, dims=[3]).unsqueeze(0)}
        with torch.amp.autocast('cuda', dtype=FLOAT_TYPE):
            with torch.no_grad():
                for model in lead_models:
                    for flip in tta:
                        use_batch = batch_tta if flip else batch
                
                        if getattr(model, "fusion_type", "") == "cross_attn":
                            with torch.no_grad():
                                with torch.amp.autocast('cuda', enabled=False):
                                    out = model({'image': use_batch['image'].float()})
                        else:
                            with torch.no_grad():
                                with torch.amp.autocast('cuda', dtype=FLOAT_TYPE):
                                    out = model(use_batch)
                
                        pixel = out['pixel'].float().cpu().numpy()[0].squeeze(1)
                
                        if not np.isfinite(pixel).all():
                            print("NON-FINITE from", getattr(model, "fusion_type", "unknown"))
                            print("min/max:", np.nanmin(pixel), np.nanmax(pixel))
                            raise ValueError("NaN/Inf detected in lead model output")
                
                        if flip:
                            pixel = np.flip(pixel, axis=flip)
                
                        for i in range(4):
                            tu, tl, lu, ll = ens_regions[i]
                            pixel_ens[i][tu:tl] += pixel[i][lu:ll]

        # --- Weighted average ---
        ens_weight = np.ones((trim_image.shape[0], trim_image.shape[1])) * len(whole_models) * len(tta)
        for i in range(4):
            tu, tl, _, _ = ens_regions[i]
            ens_weight[tu:tl] += len(lead_models) * len(tta)
        pixel_ens /= ens_weight

        try:
            # Subpixel pixel→series conversion
            series_in_pixel = pixel_to_series_exp(pixel_ens[..., t0:t1], zero_mv_trimed, length)
            series = (np.array(zero_mv_trimed).reshape(4, 1) - series_in_pixel) / mv_to_pixel
            np.save(f'{OUT_DIR}/digitalised/{sample_id}.series.npy', series)
        except:
            local_fail_id.append(sample_id)

        torch.cuda.empty_cache()

       # if n<10 and gpu_id==0: # optional: show results
         #   print()
          #  print("check max intensity : ", np.max(pixel_ens))
         #   print()
            
          #  overlay = draw_lead_pixel(trim_image, pixel_ens)
          #  plt.imshow(overlay); plt.show()
    
           # t = np.arange(len(series[0]))
          #  fig, axes = plt.subplots(4, 1, figsize=(12, 10))
           # for j in range(4):
             #   snr=0
            #    axes[j].plot(t, series[j], alpha=1.0, color='blue', linewidth=1, label='predict')
              #  axes[j].set_title(f'snr {snr:8.3f}')
              #  axes[j].legend()
         #   plt.show()
    print('')
   

    print(f'\n[GPU{gpu_id}] Stage2 done. Failed: {len(local_fail_id)}')
    if fail_id_file:
        with open(fail_id_file, 'wb') as f:
            pickle.dump(local_fail_id, f)
    return local_fail_id

run_stage2()
print('FAIL_ID:', FAIL_ID)
print('run_stage2() ok!!!\n')

In [ ]:
def split_series_to_leads(series_4xL, lead_lengths):
    """
    Reconstruct 12 leads from your saved Stage2 array exactly like make_submission().
    series_4xL shape: (4, L)
    """
    series_by_lead = {}

    lead_groups = [
        ['I',   'aVR', 'V1', 'V4'],
        ['II',  'aVL', 'V2', 'V5'],
        ['III', 'aVF', 'V3', 'V6'],
    ]

    for row_idx, group in enumerate(lead_groups):
        lengths = [int(lead_lengths[g]) for g in group]

        if group[0] == 'II':
            lengths[0] = lengths[0] - sum(lengths[1:])

        cut_idx = np.cumsum(lengths)[:-1]
        pieces = np.split(series_4xL[row_idx], cut_idx)

        for lead_name, arr in zip(group, pieces):
            series_by_lead[lead_name] = np.asarray(arr, dtype=np.float32)

    series_by_lead['II'] = np.asarray(series_4xL[3], dtype=np.float32)
    return series_by_lead


def score_one_local_sample(sample_id, out_dir=OUT_DIR, kaggle_dir=KAGGLE_DIR, train_meta_df=None):
    image_id = sample_id.split('-')[0] if '-' in sample_id else sample_id

    pred_path = f"{out_dir}/digitalised/{sample_id}.series.npy"
    truth_path = f"{kaggle_dir}/train/{image_id}/{image_id}.csv"

    if not os.path.exists(pred_path):
        raise FileNotFoundError(f"Missing prediction file: {pred_path}")
    if not os.path.exists(truth_path):
        raise FileNotFoundError(f"Missing truth file: {truth_path}")

    pred_series = np.load(pred_path)  # expected shape (4, L)
    truth_df = pd.read_csv(truth_path)

    if train_meta_df is None:
        train_meta_df = pd.read_csv(f"{kaggle_dir}/train.csv")
        train_meta_df["id"] = train_meta_df["id"].astype(str)

    meta_row = train_meta_df[train_meta_df["id"] == image_id].iloc[0]
    fs = int(meta_row["fs"])

    lead_lengths = {lead: int(truth_df[lead].count()) for lead in LEADS}
    pred_by_lead = split_series_to_leads(pred_series, lead_lengths)

    sum_signal = 0.0
    sum_noise = 0.0
    lead_scores = {}

    for lead in LEADS:
        label = truth_df[lead].dropna().to_numpy(np.float64)
        pred = np.asarray(pred_by_lead[lead], dtype=np.float64)

        if len(pred) != len(label):
            pred = scipy.signal.resample(pred, len(label)).astype(np.float64)

        aligned_pred = align_signals(label, pred, int(fs * MAX_TIME_SHIFT))
        p_signal, p_noise = compute_power(label, aligned_pred)

        sum_signal += p_signal
        sum_noise += p_noise
        lead_scores[lead] = compute_snr(p_signal, p_noise)

    image_snr_linear = compute_snr(sum_signal, sum_noise)
    image_score_db = max(float(10 * np.log10(image_snr_linear)), -PERFECT_SCORE)

    return {
        "sample_id": sample_id,
        "image_id": image_id,
        "fs": fs,
        "snr_linear": image_snr_linear,
        "score_db": image_score_db,
        **{f"{lead}_snr": lead_scores[lead] for lead in LEADS}
    }


def score_local_predictions(sample_ids=None, out_dir=OUT_DIR, kaggle_dir=KAGGLE_DIR):
    if sample_ids is None:
        sample_ids = valid_id

    train_meta_df = pd.read_csv(f"{kaggle_dir}/train.csv")
    train_meta_df["id"] = train_meta_df["id"].astype(str)

    rows = []
    failed = []

    for sample_id in sample_ids:
        try:
            rows.append(
                score_one_local_sample(
                    sample_id,
                    out_dir=out_dir,
                    kaggle_dir=kaggle_dir,
                    train_meta_df=train_meta_df
                )
            )
        except Exception as e:
            failed.append((sample_id, str(e)))

    score_df = pd.DataFrame(rows)

    if len(score_df) > 0:
        final_score_db = max(float(10 * np.log10(score_df["snr_linear"].mean())), -PERFECT_SCORE)
    else:
        final_score_db = np.nan

    print(f"Local validation score = {final_score_db:.4f}")
    print(f"Scored samples        = {len(score_df)}")
    print(f"Failed samples        = {len(failed)}")

    if failed:
        print("\nFailures:")
        for x in failed[:10]:
            print(x)

    return score_df, final_score_db, failed

In [ ]:
score_df, local_score, failed = score_local_predictions()
score_df.sort_values("score_db").head(10)

EXP_NAME = "notta_noattn_surfacefitting_1whl_1lead_model"
score_df, local_score, failed = score_local_predictions()

log_row = pd.DataFrame([{
    "exp_name": EXP_NAME,
    "local_score_db": local_score,
    "n_samples": len(score_df),
    "n_failed": len(failed),
}])

log_path = "/kaggle/working/ablation_log.csv"
if os.path.exists(log_path):
    old = pd.read_csv(log_path)
    log_row = pd.concat([old, log_row], ignore_index=True)

log_row.to_csv(log_path, index=False)
print(log_row.tail())
print(EXP_NAME)

In [ ]:
#make sbmission csv
#FAIL_ID=[1053922973, ]
def make_submission():
	print('===========================================')
	print('making submission csv ...')

	submit_df=[]
	gb = valid_df.groupby('id')
	for i,(sample_id, df) in enumerate(gb):
        
		#if sample_id in FAIL_ID:
		#	series_by_lead = {}
		#	for j,d in df.iterrows():
		#		series_by_lead[d.lead] = np.zeros(d.number_of_rows)
        
		try:
			series = np.load(f'{OUT_DIR}/digitalised/{sample_id}.series.npy')

			series_by_lead={}
			for l in range(3):
				lead = [
					['I',   'aVR', 'V1', 'V4'],
					['II',  'aVL', 'V2', 'V5'],
					['III', 'aVF', 'V3', 'V6'],
				][l]

				length=[
					df[df['lead']==lead[j]].iloc[0].number_of_rows
					for j in range(4)
				]
				if lead[0]=='II':
					length[0] = length[0]-sum(length[1:])

				index = np.cumsum(length)[:-1]
				split = np.split(series[l], index)
				#print(length)
				for (k, s) in zip(lead, split):
					series_by_lead[k] = s
					#print(k,len(s))
			series_by_lead['II'] = series[3]
			#print(series_by_lead)
    
		except: 
			series_by_lead = {}
			for j,d in df.iterrows():
				series_by_lead[d.lead] = np.zeros(d.number_of_rows)

		#print('\r\t {sample_id}', end='', flush=True)
		for j,d in df.iterrows():
			#print(d.lead, len(series_by_lead[d.lead]),d.number_of_rows)
            
			#assert(len(series_by_lead[d.lead])==d.number_of_rows)
            
            
            #probably error here ... ???
			series_by_lead[d.lead] = np.concatenate([
                series_by_lead[d.lead], np.zeros_like(series_by_lead[d.lead])
            ])[:d.number_of_rows]
			assert(len(series_by_lead[d.lead])==d.number_of_rows) 
			print(f'\r\t {i} {sample_id} : {d.lead}', end='', flush=True)

			row_id = [
				f'{sample_id}_{i}_{d.lead}' for i in range(d.number_of_rows)
			]
			this_df = pd.DataFrame({
				'id':row_id,
				'value': series_by_lead[d.lead].astype(np.float32),
			})
			submit_df.append(this_df)

	print('')
	submit_df = pd.concat(submit_df, axis=0, ignore_index=True, sort=False, copy=False)
	print(submit_df)
	submit_df.to_csv('submission.csv',index=False)

if (MODE=='fake')|(MODE=='submit'):
    make_submission()
    print('make_submission() ok!!!\n')
    if MODE=='submit':
        shutil.rmtree(OUT_DIR)
    !ls
    #!rm -rf {OUT_DIR}

'''
fake:
[21618231 rows x 2 columns]
'''

In [ ]:
sub = pd.read_csv('submission.csv')
print(sub.head())
print(sub.shape)
print('NaN count:', sub["value"].isna().sum())
print('Finite:', np.isfinite(sub["value"].to_numpy()).all())